# Week 13: Web Scraping with Python — Applied Statistics with Python (2026)

Data doesn't always come in neat CSV files. Often, the data we need lives on web pages, APIs, or online databases. This week we learn how to **collect data from the web** programmatically using Python — a critical skill for any data analyst or researcher.

| # | Topic |
|---|-------|
| 1 | How the Web Works: HTTP Basics |
| 2 | The `requests` Library |
| 3 | HTML Structure & Parsing with BeautifulSoup |
| 4 | Navigating the DOM: Finding Elements |
| 5 | Extracting Tables from Web Pages |
| 6 | Working with Web APIs (JSON) |
| 7 | Pagination & Multiple Pages |
| 8 | Handling Common Challenges |
| 9 | Ethics & Best Practices |
| 10 | Practical Example: Building a Data Pipeline |
| 11 | Summary & Homework |

## Imports

We import the core libraries for web scraping and data manipulation.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import json
import time
import re
from urllib.parse import urljoin, urlparse
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('All imports successful!')
print(f'requests version: {requests.__version__}')

---
## 1. How the Web Works: HTTP Basics

Before scraping, we need to understand how web communication works.

### The HTTP Request-Response Cycle

1. **Client** (your Python script) sends an **HTTP request** to a server
2. **Server** processes the request and sends back an **HTTP response**
3. The response contains the **HTML content** (or JSON, XML, etc.)

### HTTP Methods

| Method | Purpose | Example |
|--------|---------|---------|
| **GET** | Retrieve data (read-only) | Load a web page, fetch API data |
| **POST** | Send data to server | Submit a form, login |
| **PUT** | Update existing resource | Update a record via API |
| **DELETE** | Remove a resource | Delete a record via API |

### HTTP Status Codes

| Code | Meaning | What to do |
|------|---------|------------|
| **200** | OK — success | Parse the response |
| **301/302** | Redirect | Follow the redirect (automatic) |
| **403** | Forbidden | Check permissions, add headers |
| **404** | Not Found | URL is wrong |
| **429** | Too Many Requests | Slow down, add delays |
| **500** | Server Error | Retry later |

---
## 2. The `requests` Library

The `requests` library is the standard tool for making HTTP requests in Python. It handles connections, encoding, cookies, and redirects automatically.

### 2.1 Making a Simple GET Request

Let's fetch a web page and inspect the response object.

In [ ]:
# Fetch a simple web page
url = 'https://httpbin.org/get'  # a test endpoint that echoes back request info
response = requests.get(url)

print(f'URL: {url}')
print(f'Status code: {response.status_code}')
print(f'Content type: {response.headers["Content-Type"]}')
print(f'Encoding: {response.encoding}')
print(f'Response size: {len(response.content)} bytes')
print(f'\nResponse body (first 500 chars):')
print(response.text[:500])

### 2.2 Request Headers

Many websites check the `User-Agent` header to identify the client. Setting a proper User-Agent is important — some sites block requests without one.

In [ ]:
# Custom headers — always identify yourself properly
headers = {
    'User-Agent': 'Mozilla/5.0 (Educational Web Scraping - Statistics Course)',
    'Accept': 'text/html,application/json',
    'Accept-Language': 'en-US,en;q=0.9',
}

response = requests.get('https://httpbin.org/headers', headers=headers)
print('Server saw these headers from us:')
print(json.dumps(response.json()['headers'], indent=2))

### 2.3 Query Parameters

GET requests often include query parameters (the `?key=value&key2=value2` part of a URL). The `requests` library handles these cleanly.

In [ ]:
# Pass query parameters as a dictionary
params = {
    'q': 'python statistics',
    'page': 1,
    'limit': 10
}

response = requests.get('https://httpbin.org/get', params=params)
data = response.json()

print(f'Final URL: {response.url}')  # requests builds the URL for us
print(f'Parameters received by server: {data["args"]}')

### 2.4 POST Requests

POST requests send data to the server — commonly used for form submissions or API calls that modify data.

In [ ]:
# POST with form data
form_data = {
    'username': 'student',
    'course': 'Applied Statistics 2026'
}
response = requests.post('https://httpbin.org/post', data=form_data)
print('Form data POST:')
print(f'  Status: {response.status_code}')
print(f'  Server received: {response.json()["form"]}')

# POST with JSON data (common for APIs)
json_data = {
    'name': 'Alice',
    'scores': [85, 92, 78, 95]
}
response = requests.post('https://httpbin.org/post', json=json_data)
print(f'\nJSON POST:')
print(f'  Status: {response.status_code}')
print(f'  Server received: {response.json()["json"]}')

### 2.5 Handling Errors Gracefully

Always handle potential errors when making web requests — networks are unreliable.

In [ ]:
def safe_request(url, method='get', max_retries=3, delay=1, **kwargs):
    """
    Make an HTTP request with error handling and retries.
    
    Parameters:
    -----------
    url : target URL
    method : 'get' or 'post'
    max_retries : number of retry attempts
    delay : seconds between retries
    **kwargs : additional arguments passed to requests
    
    Returns:
    --------
    response object or None if all retries fail
    """
    for attempt in range(max_retries):
        try:
            if method == 'get':
                response = requests.get(url, timeout=10, **kwargs)
            else:
                response = requests.post(url, timeout=10, **kwargs)
            
            response.raise_for_status()  # raises exception for 4xx/5xx
            return response
            
        except requests.exceptions.Timeout:
            print(f'  Attempt {attempt+1}: Timeout. Retrying...')
        except requests.exceptions.ConnectionError:
            print(f'  Attempt {attempt+1}: Connection error. Retrying...')
        except requests.exceptions.HTTPError as e:
            print(f'  Attempt {attempt+1}: HTTP error {e}. Retrying...')
        except requests.exceptions.RequestException as e:
            print(f'  Attempt {attempt+1}: Error: {e}')
            return None
        
        time.sleep(delay)  # wait before retrying
    
    print(f'  All {max_retries} attempts failed.')
    return None

# Test with a valid URL
response = safe_request('https://httpbin.org/get')
if response:
    print(f'Success! Status: {response.status_code}')

# Test with an invalid URL
print('\nTesting with bad URL:')
response = safe_request('https://httpbin.org/status/500', max_retries=2, delay=0.5)
print(f'Result: {response}')

---
## 3. HTML Structure & Parsing with BeautifulSoup

HTML (HyperText Markup Language) is the standard language for web pages. It uses a **tree structure** of nested tags:

```html
<html>
  <head><title>Page Title</title></head>
  <body>
    <h1>Main Heading</h1>
    <div class="content">
      <p>A paragraph with <a href="url">a link</a>.</p>
      <table>
        <tr><th>Name</th><th>Score</th></tr>
        <tr><td>Alice</td><td>95</td></tr>
      </table>
    </div>
  </body>
</html>
```

**BeautifulSoup** parses this HTML into a navigable Python object.

### 3.1 Parsing HTML with BeautifulSoup

Let's parse a sample HTML document and explore the parsed tree.

In [ ]:
# Sample HTML document
html_doc = """
<html>
<head><title>Student Performance Report</title></head>
<body>
  <h1>Class Statistics — Fall 2026</h1>
  <p class="intro">This report summarizes student performance.</p>
  
  <div id="summary" class="section">
    <h2>Summary</h2>
    <ul>
      <li>Total students: <span class="stat">35</span></li>
      <li>Average score: <span class="stat">82.5</span></li>
      <li>Pass rate: <span class="stat">91.4%</span></li>
    </ul>
  </div>
  
  <div id="grades" class="section">
    <h2>Grade Distribution</h2>
    <table id="grade-table">
      <thead>
        <tr><th>Name</th><th>Score</th><th>Grade</th></tr>
      </thead>
      <tbody>
        <tr><td>Alice</td><td>95</td><td>A</td></tr>
        <tr><td>Bob</td><td>78</td><td>B</td></tr>
        <tr><td>Charlie</td><td>88</td><td>A</td></tr>
        <tr><td>Diana</td><td>62</td><td>C</td></tr>
        <tr><td>Eve</td><td>45</td><td>F</td></tr>
      </tbody>
    </table>
  </div>
  
  <div id="links" class="section">
    <h2>Resources</h2>
    <a href="https://example.com/syllabus" class="resource">Syllabus</a>
    <a href="https://example.com/textbook" class="resource">Textbook</a>
    <a href="https://example.com/office-hours" class="resource">Office Hours</a>
  </div>
</body>
</html>
"""

# Parse with BeautifulSoup
soup = BeautifulSoup(html_doc, 'html.parser')

# Basic navigation
print(f'Title: {soup.title.string}')
print(f'First h1: {soup.h1.string}')
print(f'First paragraph: {soup.p.string}')
print(f'First paragraph class: {soup.p["class"]}')

---
## 4. Navigating the DOM: Finding Elements

BeautifulSoup provides several powerful methods to find elements in the HTML tree:

| Method | Description | Returns |
|--------|-------------|----------|
| `find()` | Find first matching element | Single element or None |
| `find_all()` | Find all matching elements | List of elements |
| `select()` | CSS selector syntax | List of elements |
| `select_one()` | CSS selector, first match | Single element or None |

### 4.1 Finding Elements by Tag, Class, and ID

The `find()` and `find_all()` methods let us search by tag name, CSS class, ID, and other attributes.

In [ ]:
# Find by tag name
all_h2 = soup.find_all('h2')
print('All <h2> headings:')
for h in all_h2:
    print(f'  - {h.string}')

# Find by CSS class
stats = soup.find_all('span', class_='stat')
print(f'\nStatistics (class="stat"):')
for s in stats:
    print(f'  - {s.string}')

# Find by ID (unique on page)
grades_div = soup.find('div', id='grades')
print(f'\nGrades section heading: {grades_div.h2.string}')

# Find by attribute
links = soup.find_all('a', class_='resource')
print(f'\nResource links:')
for link in links:
    print(f'  - {link.string}: {link["href"]}')

### 4.2 CSS Selectors

CSS selectors offer a more flexible and familiar syntax for finding elements.

In [ ]:
# CSS selector syntax examples

# Select by tag
print('h2 tags:', [h.string for h in soup.select('h2')])

# Select by class (use dot notation)
print('\n.stat elements:', [s.string for s in soup.select('.stat')])

# Select by ID (use hash notation)
summary = soup.select_one('#summary')
print(f'\n#summary section exists: {summary is not None}')

# Descendant selector (elements inside others)
table_cells = soup.select('#grade-table tbody td')
print(f'\nAll table cells: {[td.string for td in table_cells]}')

# Attribute selector
resource_links = soup.select('a[class="resource"]')
print(f'\nResource links: {[a.string for a in resource_links]}')

# Combined selectors
first_row = soup.select('#grade-table tbody tr:first-child td')
print(f'\nFirst student row: {[td.string for td in first_row]}')

### 4.3 Extracting Text and Attributes

Once we find elements, we need to extract their text content or attribute values.

In [ ]:
# .string — direct text content (only works if tag has one child)
print(f'Title string: {soup.title.string}')

# .get_text() — all text content (strips nested tags)
summary_div = soup.find('div', id='summary')
print(f'\nSummary text (raw): {repr(summary_div.get_text())[:100]}...')
print(f'Summary text (clean): {summary_div.get_text(strip=True, separator=" | ")}')

# Accessing attributes
first_link = soup.find('a', class_='resource')
print(f'\nLink text: {first_link.string}')
print(f'Link href: {first_link["href"]}')
print(f'Link class: {first_link["class"]}')
print(f'Link attrs: {first_link.attrs}')  # all attributes as dict

# .get() for safe attribute access (returns None if missing)
print(f'\nData attribute: {first_link.get("data-id", "not found")}')

### 4.4 Navigating the Tree

BeautifulSoup lets us move up, down, and sideways through the HTML tree.

In [ ]:
# Children — direct children of an element
body = soup.body
direct_children = [child.name for child in body.children if child.name]
print(f'Body direct children: {direct_children}')

# Parent — go up the tree
first_td = soup.find('td')
print(f'\nFirst <td> text: {first_td.string}')
print(f'Parent tag: <{first_td.parent.name}>')
print(f'Grandparent tag: <{first_td.parent.parent.name}>')

# Siblings — next and previous at the same level
first_tr = soup.find('tbody').find('tr')
print(f'\nFirst row: {[td.string for td in first_tr.find_all("td")]}')
second_tr = first_tr.find_next_sibling('tr')
print(f'Next row:  {[td.string for td in second_tr.find_all("td")]}')

# find_next / find_previous — search forward/backward
h2_grades = soup.find('h2', string='Grade Distribution')
next_table = h2_grades.find_next('table')
print(f'\nTable after "Grade Distribution": id="{next_table["id"]}"')

---
## 5. Extracting Tables from Web Pages

Tables are one of the most common data structures on web pages. We can extract them manually with BeautifulSoup or use `pandas.read_html()` for a quick shortcut.

### 5.1 Manual Table Extraction

Let's extract the grade table from our HTML step by step.

In [ ]:
# Method 1: Manual extraction with BeautifulSoup
table = soup.find('table', id='grade-table')

# Extract headers
headers = [th.string for th in table.find('thead').find_all('th')]
print(f'Headers: {headers}')

# Extract rows
rows = []
for tr in table.find('tbody').find_all('tr'):
    row = [td.string for td in tr.find_all('td')]
    rows.append(row)

# Create DataFrame
df_manual = pd.DataFrame(rows, columns=headers)
df_manual['Score'] = df_manual['Score'].astype(int)  # convert types
print('\nExtracted table:')
print(df_manual)
print(f'\nMean score: {df_manual["Score"].mean():.1f}')

### 5.2 Quick Table Extraction with pandas

`pandas.read_html()` can automatically find and parse all HTML tables — much faster for simple cases.

In [ ]:
# Method 2: pandas.read_html() — extracts ALL tables from HTML
tables = pd.read_html(html_doc)
print(f'Found {len(tables)} table(s) in the HTML')
print('\nFirst table:')
print(tables[0])

# pd.read_html() also works directly with URLs:
# tables = pd.read_html('https://example.com/page-with-table')

### 5.3 Extracting Tables from a Live Website

Let's extract a real table from Wikipedia. We'll use a table of world population data.

In [ ]:
# Fetch Wikipedia page with population data
url = 'https://en.wikipedia.org/wiki/List_of_countries_by_population_(United_Nations)'
response = safe_request(url, headers=headers)

if response:
    # Use pandas to extract tables
    wiki_tables = pd.read_html(response.text)
    print(f'Found {len(wiki_tables)} tables on the page')
    
    # The main table is usually the largest one
    for i, t in enumerate(wiki_tables):
        print(f'  Table {i}: {t.shape[0]} rows × {t.shape[1]} cols')
    
    # Display the first few rows of the largest table
    main_table = max(wiki_tables, key=lambda t: t.shape[0])
    print(f'\nLargest table shape: {main_table.shape}')
    print(main_table.head(10))
else:
    print('Could not fetch the page. Using sample data instead.')
    main_table = pd.DataFrame({
        'Country': ['China', 'India', 'United States', 'Indonesia', 'Pakistan'],
        'Population': [1425671352, 1428627663, 339996563, 277534122, 240485658]
    })
    print(main_table)

---
## 6. Working with Web APIs (JSON)

Many websites provide **APIs** (Application Programming Interfaces) that return structured data in JSON format. APIs are the preferred way to get data — they're designed for programmatic access and return clean, structured data.

### API vs Web Scraping

| Feature | API | Web Scraping |
|---------|-----|-------------|
| Data format | Structured (JSON/XML) | Unstructured (HTML) |
| Reliability | Stable (versioned) | Fragile (HTML changes) |
| Rate limits | Clearly defined | Must guess |
| Legal status | Usually permitted | Gray area |
| Authentication | API key / OAuth | Often not needed |

### 6.1 Fetching JSON from a Public API

Let's use the JSONPlaceholder API — a free fake API for testing.

In [ ]:
# Fetch users from JSONPlaceholder API
url = 'https://jsonplaceholder.typicode.com/users'
response = requests.get(url)
users = response.json()  # parse JSON response

print(f'Status: {response.status_code}')
print(f'Number of users: {len(users)}')
print(f'\nFirst user (raw JSON):')
print(json.dumps(users[0], indent=2))

### 6.2 JSON to DataFrame

API data in JSON format converts naturally to a pandas DataFrame.

In [ ]:
# Convert to DataFrame
df_users = pd.json_normalize(users)  # handles nested JSON
print(f'Columns: {list(df_users.columns)}')
print(f'\nUser overview:')
print(df_users[['id', 'name', 'email', 'address.city', 'company.name']].to_string())

### 6.3 Fetching Related Data from an API

APIs often have related endpoints. Let's combine users with their posts.

In [ ]:
# Fetch posts for all users
posts_url = 'https://jsonplaceholder.typicode.com/posts'
posts = requests.get(posts_url).json()

df_posts = pd.DataFrame(posts)
print(f'Total posts: {len(df_posts)}')
print(f'\nPosts per user:')
posts_per_user = df_posts.groupby('userId').size().reset_index(name='num_posts')

# Merge with user info
df_summary = pd.merge(
    df_users[['id', 'name', 'company.name']],
    posts_per_user,
    left_on='id', right_on='userId'
)
print(df_summary[['name', 'company.name', 'num_posts']].to_string(index=False))

# Average title length by user
df_posts['title_length'] = df_posts['title'].str.len()
avg_title = df_posts.groupby('userId')['title_length'].mean()
print(f'\nAverage title length by user: {avg_title.mean():.1f} characters')

### 6.4 Working with a Real-World API: Open-Meteo Weather

Let's use the Open-Meteo API to fetch real weather data — no API key required.

In [ ]:
# Fetch weather data for Beijing
weather_url = 'https://api.open-meteo.com/v1/forecast'
weather_params = {
    'latitude': 39.9042,     # Beijing
    'longitude': 116.4074,
    'daily': 'temperature_2m_max,temperature_2m_min,precipitation_sum',
    'timezone': 'Asia/Shanghai',
    'forecast_days': 7
}

response = safe_request(weather_url, params=weather_params)

if response:
    weather = response.json()
    
    # Create DataFrame from daily data
    df_weather = pd.DataFrame({
        'Date': weather['daily']['time'],
        'Max Temp (°C)': weather['daily']['temperature_2m_max'],
        'Min Temp (°C)': weather['daily']['temperature_2m_min'],
        'Precipitation (mm)': weather['daily']['precipitation_sum']
    })
    df_weather['Date'] = pd.to_datetime(df_weather['Date'])
    
    print('7-Day Weather Forecast for Beijing:')
    print(df_weather.to_string(index=False))
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(df_weather['Date'], df_weather['Min Temp (°C)'],
                     df_weather['Max Temp (°C)'], alpha=0.3, color='orange')
    ax.plot(df_weather['Date'], df_weather['Max Temp (°C)'], 'ro-', label='Max')
    ax.plot(df_weather['Date'], df_weather['Min Temp (°C)'], 'bo-', label='Min')
    ax.set_title('Beijing 7-Day Temperature Forecast')
    ax.set_ylabel('Temperature (°C)')
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Could not fetch weather data.')

---
## 7. Pagination & Multiple Pages

Real-world scraping often requires fetching data across multiple pages. There are two common patterns:

1. **URL-based pagination**: `?page=1`, `?page=2`, `?offset=0&limit=10`
2. **"Next" link pagination**: follow a "Next" button/link on each page

### 7.1 URL-Based Pagination

Let's scrape multiple pages from an API with page parameters.

In [ ]:
def fetch_paginated_api(base_url, total_items, per_page=10, delay=0.5):
    """
    Fetch all items from a paginated API.
    
    Parameters:
    -----------
    base_url : API endpoint
    total_items : total number of items expected
    per_page : items per page
    delay : seconds between requests (be polite!)
    
    Returns:
    --------
    all_items : list of all fetched items
    """
    all_items = []
    total_pages = (total_items + per_page - 1) // per_page
    
    for page in range(1, total_pages + 1):
        params = {'_page': page, '_limit': per_page}
        response = requests.get(base_url, params=params)
        
        if response.status_code == 200:
            items = response.json()
            all_items.extend(items)
            print(f'  Page {page}/{total_pages}: got {len(items)} items')
        else:
            print(f'  Page {page}: Error {response.status_code}')
            break
        
        time.sleep(delay)  # respect the server
    
    return all_items

# Fetch all 100 posts from JSONPlaceholder (10 per page)
print('Fetching paginated posts...')
all_posts = fetch_paginated_api(
    'https://jsonplaceholder.typicode.com/posts',
    total_items=100, per_page=20, delay=0.2
)
print(f'\nTotal items collected: {len(all_posts)}')

### 7.2 Scraping Multiple HTML Pages

For HTML pages without an API, we follow links or construct URLs to scrape multiple pages. Here's the general pattern.

In [ ]:
def scrape_multiple_pages(base_url, page_param='page', max_pages=5, delay=1.0):
    """
    General pattern for scraping multiple HTML pages.
    
    Parameters:
    -----------
    base_url : URL template (without page parameter)
    page_param : name of the page query parameter
    max_pages : maximum number of pages to scrape
    delay : seconds between requests
    
    Returns:
    --------
    all_data : list of extracted data from all pages
    """
    all_data = []
    
    for page_num in range(1, max_pages + 1):
        # Build URL
        url = f'{base_url}?{page_param}={page_num}'
        response = safe_request(url, headers=headers)
        
        if not response:
            break
        
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract data (customize this part for each site)
        # items = soup.select('.item-class')
        # for item in items:
        #     data = {'title': item.select_one('.title').text, ...}
        #     all_data.append(data)
        
        # Check if there's a next page
        # next_link = soup.select_one('a.next-page')
        # if not next_link:
        #     break
        
        print(f'  Scraped page {page_num}')
        time.sleep(delay)  # always add delay between requests
    
    return all_data

# Note: This is a template — you would customize the extraction logic
# for each specific website you're scraping.
print('Template function defined. Customize the extraction logic per website.')
print('Key points:')
print('  1. Always add delays between requests')
print('  2. Handle errors gracefully')
print('  3. Check for a "next page" link or maximum page number')
print('  4. Save data incrementally in case of failures')

---
## 8. Handling Common Challenges

Real-world scraping involves many challenges. Here are solutions to the most common ones.

### 8.1 Using Sessions for Cookies

A `requests.Session` persists cookies and headers across multiple requests — essential for sites that require login.

In [ ]:
# Using a session — cookies persist across requests
session = requests.Session()
session.headers.update(headers)  # set default headers for all requests

# First request sets a cookie
r1 = session.get('https://httpbin.org/cookies/set/session_id/abc123')

# Subsequent requests automatically include the cookie
r2 = session.get('https://httpbin.org/cookies')
print(f'Cookies maintained across requests: {r2.json()}')

# Login pattern (general template)
# session = requests.Session()
# login_data = {'username': 'user', 'password': 'pass'}
# session.post('https://example.com/login', data=login_data)
# After login, session cookies are set
# protected_page = session.get('https://example.com/dashboard')
print('\nSession objects maintain state like a real browser.')

### 8.2 Rate Limiting and Politeness

Always be respectful of the server — send requests at a reasonable rate.

In [ ]:
import time
from functools import wraps

def rate_limited(min_interval=1.0):
    """
    Decorator that ensures a function is not called 
    more frequently than min_interval seconds.
    """
    last_call = [0.0]  # mutable to allow modification in closure
    
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            elapsed = time.time() - last_call[0]
            if elapsed < min_interval:
                time.sleep(min_interval - elapsed)
            last_call[0] = time.time()
            return func(*args, **kwargs)
        return wrapper
    return decorator

# Example: rate-limited fetch function
@rate_limited(min_interval=1.0)  # max 1 request per second
def polite_fetch(url):
    """Fetch a URL with rate limiting."""
    return requests.get(url, headers=headers, timeout=10)

# Demonstration
print('Fetching 3 URLs with 1-second rate limit:')
for i in range(3):
    start = time.time()
    r = polite_fetch('https://httpbin.org/get')
    elapsed = time.time() - start
    print(f'  Request {i+1}: {r.status_code} ({elapsed:.2f}s)')

### 8.3 Parsing with Regular Expressions

Sometimes data is embedded in text (not structured HTML). Regular expressions can extract patterns.

In [ ]:
# Extracting patterns from text
sample_text = """
Contact us at info@example.com or support@company.org.
Revenue was $1,234,567 in 2025 and $2,345,678 in 2026.
Phone: (555) 123-4567 or 555-987-6543.
Visit https://www.example.com/data or http://api.test.io/v2.
"""

# Extract emails
emails = re.findall(r'[\w.+-]+@[\w-]+\.[\w.]+', sample_text)
print(f'Emails: {emails}')

# Extract dollar amounts
amounts = re.findall(r'\$[\d,]+', sample_text)
print(f'Dollar amounts: {amounts}')

# Extract phone numbers
phones = re.findall(r'\(?\d{3}\)?[\s-]?\d{3}[\s-]?\d{4}', sample_text)
print(f'Phone numbers: {phones}')

# Extract URLs
urls = re.findall(r'https?://[\w./\-]+', sample_text)
print(f'URLs: {urls}')

# Extract years
years = re.findall(r'\b20\d{2}\b', sample_text)
print(f'Years: {years}')

### 8.4 Checking robots.txt

Before scraping any website, always check its `robots.txt` file to see what's allowed.

In [ ]:
def check_robots_txt(base_url):
    """
    Fetch and display a website's robots.txt file.
    This tells us what we're allowed to scrape.
    """
    parsed = urlparse(base_url)
    robots_url = f'{parsed.scheme}://{parsed.netloc}/robots.txt'
    
    response = safe_request(robots_url)
    if response:
        print(f'robots.txt for {parsed.netloc}:')
        print('-' * 50)
        # Show first 30 lines
        lines = response.text.strip().split('\n')[:30]
        for line in lines:
            print(line)
        if len(response.text.strip().split('\n')) > 30:
            print('... (truncated)')
    else:
        print(f'Could not fetch robots.txt for {parsed.netloc}')

# Check robots.txt for a well-known site
check_robots_txt('https://en.wikipedia.org')

---
## 9. Ethics & Best Practices

Web scraping comes with ethical and legal responsibilities. Follow these guidelines:

### Rules of Responsible Scraping

| Rule | Description |
|------|-------------|
| **Check robots.txt** | Respect the site's crawling rules |
| **Read Terms of Service** | Some sites explicitly prohibit scraping |
| **Use APIs when available** | Prefer structured access over scraping |
| **Rate limit your requests** | Don't overwhelm servers (1-2 req/sec max) |
| **Identify yourself** | Set a descriptive User-Agent |
| **Don't scrape personal data** | Respect privacy laws (GDPR, etc.) |
| **Cache responses** | Don't re-fetch the same page repeatedly |
| **Handle errors gracefully** | Don't retry failed requests aggressively |
| **Give credit** | Cite your data sources |

### 9.1 Building a Polite Scraper Class

Let's create a reusable scraper class that follows all best practices.

In [ ]:
class PoliteScraper:
    """
    A well-behaved web scraper that follows best practices.
    
    Features:
    - Rate limiting between requests
    - Automatic retries with backoff
    - Session management for cookies
    - Response caching
    - Custom User-Agent
    """
    
    def __init__(self, delay=1.0, max_retries=3, user_agent=None):
        self.delay = delay
        self.max_retries = max_retries
        self.session = requests.Session()
        self.session.headers['User-Agent'] = (
            user_agent or 'PoliteScraper/1.0 (Educational Purpose)'
        )
        self.cache = {}  # simple URL → response cache
        self.last_request_time = 0
        self.request_count = 0
    
    def _wait(self):
        """Enforce rate limiting."""
        elapsed = time.time() - self.last_request_time
        if elapsed < self.delay:
            time.sleep(self.delay - elapsed)
    
    def get(self, url, use_cache=True, **kwargs):
        """
        Fetch a URL with rate limiting, caching, and retries.
        """
        # Check cache
        if use_cache and url in self.cache:
            return self.cache[url]
        
        # Rate limit
        self._wait()
        
        # Retry loop
        for attempt in range(self.max_retries):
            try:
                response = self.session.get(url, timeout=15, **kwargs)
                self.last_request_time = time.time()
                self.request_count += 1
                
                if response.status_code == 429:  # rate limited
                    wait = int(response.headers.get('Retry-After', 60))
                    print(f'  Rate limited. Waiting {wait}s...')
                    time.sleep(wait)
                    continue
                
                response.raise_for_status()
                
                # Cache successful responses
                if use_cache:
                    self.cache[url] = response
                
                return response
                
            except requests.RequestException as e:
                if attempt < self.max_retries - 1:
                    wait = 2 ** attempt  # exponential backoff
                    print(f'  Error: {e}. Retrying in {wait}s...')
                    time.sleep(wait)
        
        return None
    
    def get_soup(self, url, **kwargs):
        """Fetch a URL and return a BeautifulSoup object."""
        response = self.get(url, **kwargs)
        if response:
            return BeautifulSoup(response.text, 'html.parser')
        return None
    
    def stats(self):
        """Print scraping statistics."""
        print(f'Total requests: {self.request_count}')
        print(f'Cached pages: {len(self.cache)}')

# Demo
scraper = PoliteScraper(delay=0.5)
response = scraper.get('https://httpbin.org/get')
if response:
    print(f'Fetched successfully: {response.status_code}')

# Second request to same URL uses cache
response2 = scraper.get('https://httpbin.org/get')
print(f'Second request (cached): {response2.status_code}')

scraper.stats()

---
## 10. Practical Example: Building a Data Pipeline

Let's build a complete data collection pipeline that fetches data from multiple sources, cleans it, and creates a dataset for analysis. We'll collect country data from a public API and enrich it with additional information.

### Step 1: Fetch Country Data from REST Countries API

In [ ]:
# Fetch country data from REST Countries API
countries_url = 'https://restcountries.com/v3.1/all'
params = {'fields': 'name,population,area,region,subregion,capital,languages,currencies'}

response = safe_request(countries_url, params=params)

if response:
    countries_raw = response.json()
    print(f'Fetched data for {len(countries_raw)} countries')
    print(f'\nSample entry:')
    print(json.dumps(countries_raw[0], indent=2, ensure_ascii=False)[:500])
else:
    # Fallback sample data
    print('API unavailable. Using sample data.')
    countries_raw = [
        {'name': {'common': 'China'}, 'population': 1425671352, 'area': 9596961, 
         'region': 'Asia', 'subregion': 'Eastern Asia', 'capital': ['Beijing']},
        {'name': {'common': 'India'}, 'population': 1428627663, 'area': 3287263,
         'region': 'Asia', 'subregion': 'Southern Asia', 'capital': ['New Delhi']},
        {'name': {'common': 'United States'}, 'population': 339996563, 'area': 9833520,
         'region': 'Americas', 'subregion': 'North America', 'capital': ['Washington, D.C.']},
        {'name': {'common': 'Brazil'}, 'population': 216422446, 'area': 8515767,
         'region': 'Americas', 'subregion': 'South America', 'capital': ['Brasília']},
        {'name': {'common': 'Nigeria'}, 'population': 223804632, 'area': 923768,
         'region': 'Africa', 'subregion': 'Western Africa', 'capital': ['Abuja']},
    ]

### Step 2: Clean and Structure the Data

Raw API data often needs cleaning before analysis — nested fields, missing values, and inconsistent formats.

In [ ]:
# Parse nested JSON into a clean DataFrame
clean_data = []

for country in countries_raw:
    try:
        name = country.get('name', {}).get('common', 'Unknown')
        population = country.get('population', 0)
        area = country.get('area', 0)
        region = country.get('region', 'Unknown')
        subregion = country.get('subregion', 'Unknown')
        
        # Capital can be a list or missing
        capitals = country.get('capital', [])
        capital = capitals[0] if capitals else 'N/A'
        
        # Languages is a dict {code: name}
        languages = country.get('languages', {})
        num_languages = len(languages)
        primary_language = list(languages.values())[0] if languages else 'N/A'
        
        # Population density
        density = population / area if area > 0 else 0
        
        clean_data.append({
            'Country': name,
            'Capital': capital,
            'Region': region,
            'Subregion': subregion,
            'Population': population,
            'Area_km2': area,
            'Density_per_km2': round(density, 1),
            'Num_Languages': num_languages,
            'Primary_Language': primary_language
        })
    except (KeyError, IndexError, TypeError):
        continue  # skip problematic entries

df = pd.DataFrame(clean_data)
print(f'Cleaned dataset: {df.shape[0]} countries, {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
print(f'\nSample (top 10 by population):')
print(df.nlargest(10, 'Population')[['Country', 'Region', 'Population', 
                                      'Area_km2', 'Density_per_km2']].to_string(index=False))

### Step 3: Exploratory Analysis

Now let's analyze the scraped data with statistics and visualizations.

In [ ]:
# Filter out tiny countries for meaningful analysis
df_main = df[df['Population'] > 100000].copy()
print(f'Countries with population > 100K: {len(df_main)}')

# Summary statistics by region
region_stats = df_main.groupby('Region').agg(
    Countries=('Country', 'count'),
    Total_Pop=('Population', 'sum'),
    Avg_Pop=('Population', 'mean'),
    Avg_Area=('Area_km2', 'mean'),
    Avg_Density=('Density_per_km2', 'mean')
).round(0)

region_stats['Total_Pop_M'] = (region_stats['Total_Pop'] / 1e6).round(0)
print('\nRegion Summary:')
print(region_stats[['Countries', 'Total_Pop_M', 'Avg_Density']].to_string())

### Step 4: Visualization Dashboard

Let's create a comprehensive visualization of our scraped data.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Population by region (bar chart)
region_pop = df_main.groupby('Region')['Population'].sum().sort_values(ascending=True)
region_pop_b = region_pop / 1e9  # billions
region_pop_b.plot(kind='barh', ax=axes[0,0], color='steelblue')
axes[0,0].set_title('Total Population by Region (Billions)')
axes[0,0].set_xlabel('Population (Billions)')

# 2. Population density distribution (histogram)
df_plot = df_main[df_main['Density_per_km2'] < 1000]  # exclude extreme outliers
axes[0,1].hist(df_plot['Density_per_km2'], bins=40, color='coral', edgecolor='white')
axes[0,1].axvline(df_plot['Density_per_km2'].median(), color='red', ls='--',
                    label=f'Median: {df_plot["Density_per_km2"].median():.0f}')
axes[0,1].set_title('Distribution of Population Density')
axes[0,1].set_xlabel('People per km²')
axes[0,1].set_ylabel('Number of Countries')
axes[0,1].legend()

# 3. Population vs Area (scatter)
for region in df_main['Region'].unique():
    mask = df_main['Region'] == region
    axes[1,0].scatter(df_main.loc[mask, 'Area_km2'] / 1e6,
                       df_main.loc[mask, 'Population'] / 1e6,
                       label=region, alpha=0.6, s=30)
axes[1,0].set_xlabel('Area (million km²)')
axes[1,0].set_ylabel('Population (millions)')
axes[1,0].set_title('Population vs Area by Region')
axes[1,0].legend(fontsize=8, loc='upper right')

# 4. Top 15 most populated countries
top15 = df_main.nlargest(15, 'Population')
axes[1,1].barh(top15['Country'], top15['Population'] / 1e6, color='teal')
axes[1,1].set_xlabel('Population (millions)')
axes[1,1].set_title('Top 15 Most Populated Countries')
axes[1,1].invert_yaxis()

plt.suptitle('World Country Data — Collected via Web API', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Step 5: Save the Dataset

Always save your scraped data to avoid re-fetching.

In [ ]:
# Save to CSV
output_file = 'world_countries_data.csv'
df.to_csv(output_file, index=False)
print(f'Dataset saved to {output_file}')
print(f'  Rows: {len(df)}')
print(f'  Columns: {len(df.columns)}')
print(f'  File size: {len(df.to_csv(index=False)):,} bytes')

# Data quality report
print(f'\n=== Data Quality Report ===')
print(f'Missing values:')
for col in df.columns:
    missing = df[col].isna().sum() + (df[col] == 'N/A').sum() + (df[col] == 0).sum()
    if missing > 0:
        print(f'  {col}: {missing} ({missing/len(df)*100:.1f}%)')

print(f'\nData collection complete!')

---
## 11. Summary

| Section | Key Concepts | Python Tools |
|---------|-------------|---------------|
| 1. HTTP Basics | Request-response cycle, methods, status codes | Conceptual |
| 2. `requests` | GET/POST, headers, params, error handling | `requests.get()`, `.post()`, `.json()` |
| 3. HTML & BeautifulSoup | HTML tree structure, parsing | `BeautifulSoup()`, `html.parser` |
| 4. Finding Elements | find/find_all, CSS selectors, navigation | `.find()`, `.select()`, `.get_text()` |
| 5. Table Extraction | Manual vs pandas approach | `pd.read_html()`, manual parsing |
| 6. Web APIs | JSON endpoints, REST, data merging | `.json()`, `pd.json_normalize()` |
| 7. Pagination | Multi-page scraping, URL patterns | Loops with `time.sleep()` |
| 8. Challenges | Sessions, rate limiting, regex parsing | `requests.Session()`, `re.findall()` |
| 9. Ethics | robots.txt, ToS, rate limits, privacy | `check_robots_txt()`, `PoliteScraper` |
| 10. Data Pipeline | End-to-end collection, cleaning, analysis | Full workflow |

## Homework

### Task 1: API Data Collection
Use the Open-Meteo API (`https://api.open-meteo.com/v1/forecast`) to:
1. Fetch 7-day weather forecasts for 5 different cities of your choice.
2. Combine all data into a single DataFrame with a `City` column.
3. Create a comparison plot showing temperature trends across all cities.
4. Which city has the highest average temperature? The most precipitation?

### Task 2: Wikipedia Table Scraping
Scrape a data table from any Wikipedia page of your choice (e.g., list of universities, Olympic medals, etc.):
1. Use `requests` + `pd.read_html()` to extract the table.
2. Clean the data (handle missing values, fix data types).
3. Perform basic statistical analysis (mean, median, distribution).
4. Create at least 2 meaningful visualizations.

### Task 3: Build a Reusable Scraper
Extend the `PoliteScraper` class to add:
1. A `save_cache(filename)` method that saves the cache to a JSON file.
2. A `load_cache(filename)` method that loads a previously saved cache.
3. A `clear_cache()` method.
4. Test your improved scraper by fetching data from the JSONPlaceholder API.

### Task 4: End-to-End Data Pipeline
Build a complete data collection pipeline:
1. Choose a public API (suggestions: https://github.com/public-apis/public-apis).
2. Fetch at least 100 records from the API.
3. Clean and structure the data into a pandas DataFrame.
4. Perform exploratory data analysis (5 number summary, distributions, correlations).
5. Save the data to CSV and create a summary report with visualizations.

---
## Next Week Preview

**Week 14: Machine Learning Fundamentals & Supervised Learning** — We'll enter the world of machine learning! Topics include the ML workflow, train/test splitting, k-Nearest Neighbors, Decision Trees, Random Forests, and model evaluation with scikit-learn.

---
*Applied Statistics with Python (2026) | Week 13 | Web Scraping with Python*